# 01 — Exploratory Data Analysis

Beijing air-quality (UCI #501) — station `Aotizhongxin`, target `PM2.5`.
Saves all figures to `docs/images/`. Run via `make eda`.

In [1]:
import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path

import probforecast
from probforecast.config import load_config
from probforecast.data.schema import TARGET, COVARIATES

# nbconvert executes with cwd=notebooks/; relative config paths assume project root.
os.chdir(Path(probforecast.__file__).resolve().parents[2])

cfg = load_config()
IMG = Path(cfg.paths.images_dir)
IMG.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

proc = pd.read_parquet(Path(cfg.data.processed_dir) / f"{cfg.data.station}.parquet")
proc["timestamp"] = pd.to_datetime(proc["timestamp"])
proc = proc.set_index("timestamp")
print(proc.shape)
proc[[TARGET]].describe()

(35064, 31)


,PM2.5
count,34454.000000
mean,82.800670
std,82.285306
min,3.000000
25%,22.000000
50%,58.000000
75%,114.000000
max,898.000000


## 1. Full series with train/val/test boundaries

In [2]:
s = cfg.data.split
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(proc.index, proc[TARGET], lw=0.4, color="#333")
for d, c, lab in [
    (s.val_start, "tab:orange", "val start"),
    (s.test_start, "tab:red", "test start"),
]:
    ax.axvline(pd.Timestamp(d), color=c, ls="--", label=lab)
ax.set(title=f"PM2.5 — {cfg.data.station}", ylabel="PM2.5 (\u00b5g/m\u00b3)")
ax.legend()
fig.tight_layout()
fig.savefig(IMG / "eda_series.png", dpi=120)
print("saved eda_series.png")

saved eda_series.png


## 2. Daily & weekly seasonality profiles

In [3]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
by_hour = proc.groupby(proc.index.hour)[TARGET].mean()
by_dow = proc.groupby(proc.index.dayofweek)[TARGET].mean()
axes[0].plot(by_hour.index, by_hour.values, marker="o")
axes[0].set(title="Mean PM2.5 by hour of day", xlabel="hour", ylabel="PM2.5")
axes[1].plot(by_dow.index, by_dow.values, marker="o", color="tab:green")
axes[1].set(title="Mean PM2.5 by day of week", xlabel="0=Mon \u2026 6=Sun", ylabel="PM2.5")
fig.tight_layout()
fig.savefig(IMG / "eda_seasonality.png", dpi=120)
print("saved eda_seasonality.png")

saved eda_seasonality.png


## 3. Target distribution (heavy-tailed? log helps?)

In [4]:
vals = proc[TARGET].dropna()
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(vals, bins=60, color="#555")
axes[0].set(title="PM2.5 distribution", xlabel="PM2.5", ylabel="count")
axes[1].hist(np.log1p(vals), bins=60, color="tab:blue")
axes[1].set(title="log1p(PM2.5)", xlabel="log1p(PM2.5)")
fig.tight_layout()
fig.savefig(IMG / "eda_distribution.png", dpi=120)
print("saved eda_distribution.png")

saved eda_distribution.png


## 4. Missingness heatmap (raw target, by year \u00d7 month)

In [5]:
raw = pd.read_csv(Path(cfg.data.raw_dir) / f"PRSA_Data_{cfg.data.station}.csv")
raw["timestamp"] = pd.to_datetime(raw[["year", "month", "day", "hour"]])
raw = raw.set_index("timestamp")
miss = raw[TARGET].isna().groupby([raw.index.year, raw.index.month]).mean().unstack()
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(miss, cmap="rocket_r", ax=ax, cbar_kws={"label": "fraction missing"})
ax.set(title="Raw PM2.5 missingness by month", xlabel="month", ylabel="year")
fig.tight_layout()
fig.savefig(IMG / "eda_missingness.png", dpi=120)
print("saved eda_missingness.png")

saved eda_missingness.png


## 5. Covariate correlation matrix

In [6]:
cols = [TARGET] + [c for c in COVARIATES if c in proc.columns]
corr = proc[cols].corr()
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set(title="Correlation: target + covariates")
fig.tight_layout()
fig.savefig(IMG / "eda_correlation.png", dpi=120)
print("saved eda_correlation.png")

saved eda_correlation.png
